In [ ]:
"""
Computation of global correlations by subject group

Script for computing and visualizing HRV/PRV correlations across subject groups.

Main functionalities:
- Allows selection of the analysis method ('All Windows' or 'Method 2').
- Computes Pearson and Spearman correlations between HRV and PRV metrics for each subject.
- Aggregates results by group (Healthy, RBD, OSAS, PLM, Mixed).
- Displays grouped bar charts showing mean correlation values for each HRV metric across groups.
"""


# =============================================================================
# IMPORT LIBRARIES
# =============================================================================

import os
import re
import pandas as pd
import matplotlib.pyplot as plt

from utils_analysis import compute_correlations_for_metrics

from utils_analysis_plot import plot_grouped_bars

%matplotlib qt


# =============================================================================
# SETTINGS
# =============================================================================
method_name = "Method 2"  # or "Method 2"

base_path = os.path.join("..", "Data")
methods = {
    "All Windows": os.path.join(base_path, "All Windows"),
    "Method 2": os.path.join(base_path, "Window Removal")
}


# =============================================================================
# SUBJECT LIST AND GROUP ASSIGNMENT
# =============================================================================

# List subjects based on HRV folder of 'All Windows'
subjects_dir = os.path.join(methods["All Windows"], "HRV")
subjects = [f for f in os.listdir(subjects_dir) if f.endswith(".csv")]
subjects = sorted(subjects, key=lambda x: int(re.findall(r'\d+', x)[0]))  # sort numerically

# Assign subjects to groups
groups = ["HC"] * 10 + ["RBD"] * 10 + ["OSAS"] * 10 + ["PLM"] * 10 + ["mixed"] * 10
subject_to_group = dict(zip(subjects, groups))

# Initialize dictionaries to store correlation values per group
group_pearson = {"HC": [], "RBD": [], "OSAS": [], "PLM": [], "mixed": []}
group_spearman = {"HC": [], "RBD": [], "OSAS": [], "PLM": [], "mixed": []}


# =============================================================================
# COMPUTE CORRELATIONS
# =============================================================================

for subj_file in subjects:
    group = subject_to_group[subj_file]

    # Load HRV and PRV data
    hrv_path = os.path.join(methods[method_name], "HRV", subj_file)
    prv_path = os.path.join(methods[method_name], "PRV", subj_file)

    df_hrv = pd.read_csv(hrv_path).drop(columns=["Window", "HRV_RMSSD"], errors="ignore")
    df_prv = pd.read_csv(prv_path).drop(columns=["Window", "HRV_RMSSD"], errors="ignore")

    # Compute Pearson and Spearman correlations for all metrics
    pearson, spearman = compute_correlations_for_metrics(df_hrv, df_prv)

    # Store results by group
    group_pearson[group].append(pearson)
    group_spearman[group].append(spearman)


# =============================================================================
# MEAN AND STD CORRELATIONS FOR EACH GROUP
# =============================================================================

# Define group and metrics
group_names = list(group_pearson.keys())
metrics = list(group_pearson[group_names[0]][0].keys())

# Initialize dictionaries for plotting
pearson_mean = {"Metric": metrics}
pearson_std  = {"Metric": metrics}

spearman_mean = {"Metric": metrics}
spearman_std  = {"Metric": metrics}

# Compute mean and std correlations across subjects for each group
for group in group_names:
    df_p = pd.DataFrame(group_pearson[group])
    df_s = pd.DataFrame(group_spearman[group])

    # Pearson
    pearson_mean[group] = df_p.mean().values
    pearson_std[group]  = df_p.std().values

    # Spearman
    spearman_mean[group] = df_s.mean().values
    spearman_std[group]  = df_s.std().values

# Convert to DataFrames 
df_pearson_mean = pd.DataFrame(pearson_mean)
df_pearson_std = pd.DataFrame(pearson_std)

df_spearman_mean = pd.DataFrame(spearman_mean)
df_spearman_std = pd.DataFrame(spearman_std)


# =============================================================================
# PLOT BARPLOT CORRELATIONS
# =============================================================================

fig, axes = plt.subplots(nrows=2, figsize=(24, 14))
#fig.suptitle(f"{method_name}", fontsize=26, fontweight="bold")

# Plot Pearson and Spearman grouped bar plots
plot_grouped_bars(axes[0], df_pearson_mean, df_pearson_std, group_names, "Pearson Correlation")
plot_grouped_bars(axes[1], df_spearman_mean, df_spearman_std, group_names, "Spearman Correlation")
#axes[0].set_xticklabels([])

fig.legend(group_names, loc='upper right', bbox_to_anchor=(0.98, 1.04), fontsize=20, ncol=5)
plt.tight_layout()
plt.show()
#plt.savefig("C:/Users/ilari/Documents/GitHub/SleepProject/Img and Results/barplot_correlations_group_method 2_v3.png", dpi=600, bbox_inches='tight') 
